<a href="https://colab.research.google.com/github/yuke313/514_project/blob/main/541.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install ortools

In [5]:
from __future__ import annotations

from collections import defaultdict
from typing import Any, Dict, List, Tuple
from ortools.sat.python import cp_model


def build_sample_data() -> Dict[str, Any]:
    """
    Small demo dataset for Phase 1.
    You can replace this with JSON-loaded data later.
    """
    return {
        "staff": [
            {
                "id": "E01",
                "name": "Amy",
                "skills": ["server"],
                "max_shifts": 2,
            },
            {
                "id": "E02",
                "name": "John",
                "skills": ["server", "cashier"],
                "max_shifts": 3,
            },
            {
                "id": "E03",
                "name": "Betty",
                "skills": ["cashier"],
                "max_shifts": 2,
            },
            {
                "id": "E04",
                "name": "Cathy",
                "skills": ["server"],
                "max_shifts": 2,
            },
            {
                "id": "E05",
                "name": "David",
                "skills": ["server", "cashier"],
                "max_shifts": 3,
            },
        ],
        "availability": [
            {"employee_id": "E01", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E01", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E01", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E01", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E02", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E02", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E02", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E02", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E03", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E03", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E03", "day": "Tuesday", "shift": "morning", "available": False},
            {"employee_id": "E03", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E04", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E04", "day": "Monday", "shift": "evening", "available": False},
            {"employee_id": "E04", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E04", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E05", "day": "Monday", "shift": "morning", "available": False},
            {"employee_id": "E05", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E05", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E05", "day": "Tuesday", "shift": "evening", "available": True},
        ],
        "shift_requirements": [
            # Monday
            {"day": "Monday", "shift": "morning", "role": "server", "required_count": 1},
            {"day": "Monday", "shift": "morning", "role": "cashier", "required_count": 1},
            {"day": "Monday", "shift": "evening", "role": "server", "required_count": 2},
            {"day": "Monday", "shift": "evening", "role": "cashier", "required_count": 1},

            # Tuesday
            {"day": "Tuesday", "shift": "morning", "role": "server", "required_count": 1},
            {"day": "Tuesday", "shift": "morning", "role": "cashier", "required_count": 1},
            {"day": "Tuesday", "shift": "evening", "role": "server", "required_count": 1},
            {"day": "Tuesday", "shift": "evening", "role": "cashier", "required_count": 1},
        ],
    }


def make_availability_lookup(availability: List[Dict[str, Any]]) -> Dict[Tuple[str, str, str], bool]:
    lookup: Dict[Tuple[str, str, str], bool] = {}
    for row in availability:
        key = (row["employee_id"], row["day"], row["shift"])
        lookup[key] = bool(row["available"])
    return lookup


def get_staff_lookup(staff: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    return {s["id"]: s for s in staff}


def generate_schedule(data: Dict[str, Any]) -> List[Dict[str, str]]:
    """
    Build a schedule from scratch with OR-Tools CP-SAT.

    Hard constraints:
    1. Required coverage must be met exactly.
    2. Employee must be available for assigned shift.
    3. Employee must have the required skill/role.
    4. Employee cannot hold two roles in the same day/shift.
    5. Employee cannot exceed max_shifts.

    Soft objective:
    - Minimize the maximum number of assigned shifts to keep load more balanced.
    """
    staff = data["staff"]
    availability = data["availability"]
    requirements = data["shift_requirements"]

    staff_lookup = get_staff_lookup(staff)
    avail_lookup = make_availability_lookup(availability)

    model = cp_model.CpModel()

    # Decision variable:
    # x[(emp_id, day, shift, role)] = 1 if employee is assigned to that role in that shift.
    x: Dict[Tuple[str, str, str, str], cp_model.IntVar] = {}

    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]

        for emp in staff:
            emp_id = emp["id"]
            emp_skills = set(emp["skills"])
            is_available = avail_lookup.get((emp_id, day, shift), False)
            has_skill = role in emp_skills

            var = model.NewBoolVar(f"x_{emp_id}_{day}_{shift}_{role}")
            x[(emp_id, day, shift, role)] = var

            # Not available or not qualified -> cannot assign
            if (not is_available) or (not has_skill):
                model.Add(var == 0)

    # Coverage constraints: exactly meet required_count
    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]
        required_count = int(req["required_count"])

        eligible_vars = [
            x[(emp["id"], day, shift, role)]
            for emp in staff
        ]
        model.Add(sum(eligible_vars) == required_count)

    # Same employee cannot take two roles in the same shift
    unique_shift_keys = sorted({(r["day"], r["shift"]) for r in requirements})
    roles = sorted({r["role"] for r in requirements})

    for emp in staff:
        emp_id = emp["id"]
        for day, shift in unique_shift_keys:
            vars_same_shift = [
                x[(emp_id, day, shift, role)]
                for role in roles
                if (emp_id, day, shift, role) in x
            ]
            if vars_same_shift:
                model.Add(sum(vars_same_shift) <= 1)

    # Max shifts per employee
    for emp in staff:
        emp_id = emp["id"]
        max_shifts = int(emp["max_shifts"])
        emp_vars = [
            var for key, var in x.items()
            if key[0] == emp_id
        ]
        model.Add(sum(emp_vars) <= max_shifts)

    # Soft fairness objective:
    # minimize the maximum assigned shifts among employees
    total_assigned_per_emp: Dict[str, cp_model.IntVar] = {}
    for emp in staff:
        emp_id = emp["id"]
        total = model.NewIntVar(0, len(unique_shift_keys), f"total_{emp_id}")
        emp_vars = [
            var for key, var in x.items()
            if key[0] == emp_id
        ]
        model.Add(total == sum(emp_vars))
        total_assigned_per_emp[emp_id] = total

    max_load = model.NewIntVar(0, len(unique_shift_keys), "max_load")
    for emp_id, total in total_assigned_per_emp.items():
        model.Add(total <= max_load)

    model.Minimize(max_load)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 10

    status = solver.Solve(model)

    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        raise ValueError("No feasible schedule found with current constraints.")

    schedule: List[Dict[str, str]] = []
    for (emp_id, day, shift, role), var in x.items():
        if solver.Value(var) == 1:
            schedule.append(
                {
                    "employee_id": emp_id,
                    "employee_name": staff_lookup[emp_id]["name"],
                    "day": day,
                    "shift": shift,
                    "role": role,
                }
            )

    schedule.sort(key=lambda r: (r["day"], r["shift"], r["role"], r["employee_name"]))
    return schedule


def print_schedule(schedule: List[Dict[str, str]]) -> None:
    grouped: Dict[Tuple[str, str], List[Dict[str, str]]] = defaultdict(list)
    for row in schedule:
        grouped[(row["day"], row["shift"])].append(row)

    print("\n===== SCHEDULE =====")
    for day_shift in sorted(grouped.keys()):
        day, shift = day_shift
        print(f"\n{day} - {shift}")
        for row in sorted(grouped[day_shift], key=lambda r: (r["role"], r["employee_name"])):
            print(f"  {row['role']:<8} -> {row['employee_name']} ({row['employee_id']})")
    print("====================\n")


def update_schedule(
    data: Dict[str, Any],
    existing_schedule: List[Dict[str, str]],
    change: Dict[str, str],
) -> List[Dict[str, str]]:
    """
    Simple Phase 1 update:
    supports requests like:
    {
        "type": "unavailable",
        "employee_name": "Amy",
        "day": "Monday",
        "shift": "evening"
    }

    Strategy:
    1. Mark that employee unavailable for the target shift.
    2. Re-run schedule generation from scratch.
       This is the simplest robust Phase 1 approach.
    """
    if change.get("type") != "unavailable":
        raise ValueError("Currently only 'unavailable' updates are supported.")

    employee_name = change["employee_name"]
    day = change["day"]
    shift = change["shift"]

    # Find employee id
    matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
    if not matched:
        raise ValueError(f"Employee '{employee_name}' not found.")

    emp_id = matched[0]["id"]

    # Copy data deeply enough for this use case
    new_data = {
        "staff": [dict(s) for s in data["staff"]],
        "availability": [dict(a) for a in data["availability"]],
        "shift_requirements": [dict(r) for r in data["shift_requirements"]],
    }

    found_availability_row = False
    for row in new_data["availability"]:
        if row["employee_id"] == emp_id and row["day"] == day and row["shift"] == shift:
            row["available"] = False
            found_availability_row = True

    if not found_availability_row:
        new_data["availability"].append(
            {
                "employee_id": emp_id,
                "day": day,
                "shift": shift,
                "available": False,
            }
        )

    # Regenerate whole schedule for simplicity
    updated_schedule = generate_schedule(new_data)
    return updated_schedule


def main() -> None:
    data = build_sample_data()

    print("Generating initial schedule...")
    schedule = generate_schedule(data)
    print_schedule(schedule)

    # Manual Phase 1 test: Amy unavailable Monday evening
    change_request = {
        "type": "unavailable",
        "employee_name": "Amy",
        "day": "Monday",
        "shift": "evening",
    }

    print("Applying update: Amy unavailable Monday evening...")
    updated = update_schedule(data, schedule, change_request)
    print_schedule(updated)


if __name__ == "__main__":
    main()

Generating initial schedule...

===== SCHEDULE =====

Monday - evening
  cashier  -> Betty (E03)
  server   -> Amy (E01)
  server   -> John (E02)

Monday - morning
  cashier  -> Betty (E03)
  server   -> Cathy (E04)

Tuesday - evening
  cashier  -> David (E05)
  server   -> Cathy (E04)

Tuesday - morning
  cashier  -> John (E02)
  server   -> David (E05)

Applying update: Amy unavailable Monday evening...

===== SCHEDULE =====

Monday - evening
  cashier  -> Betty (E03)
  server   -> David (E05)
  server   -> John (E02)

Monday - morning
  cashier  -> Betty (E03)
  server   -> Amy (E01)

Tuesday - evening
  cashier  -> John (E02)
  server   -> Cathy (E04)

Tuesday - morning
  cashier  -> David (E05)
  server   -> Cathy (E04)

